# MP2 Part 1 — OSS commit-data retrieval (`hsaleh5`)

This notebook identifies my ten assigned projects, documents the retrieval method, loads the completed semicolon-delimited data, validates it, and reports the required World of Code and GitHub statistics.

## Sources and method

[World of Code](https://worldofcode.org/) is research infrastructure that indexes and cross-links Git objects from many public repositories. It is an archival/aggregated secondary data source, whereas each GitHub repository is the live primary source for its current stars, forks, and default-branch history.

I retrieved each assigned project's authoritative SHA-1 list with WoC's `p2c` map. Commit content was processed in batches of 10 (below the assignment's limit of 50 and equal to the WoC batch endpoint's maximum), with pauses between requests and retries. To reduce load on WoC, identical SHA-addressed commit objects already present in a blobless GitHub mirror were read locally; only missing objects were requested from WoC's `commit.tch` map. The complete reproducible client is in `retrieve_part1.py`.

GitHub metadata and WoC data were collected on **2026-09-30 01:34:24 UTC**. Counts can change after that timestamp.

In [1]:
# Dependencies used by the retrieval and validation workflow.
# On Colab/Linux, the official client can also be installed with:
# %pip install -U python-woc pandas requests tqdm
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

NETID = 'hsaleh5'
ROOT = Path.cwd()
SUMMARY_FILE = ROOT / f'{NETID}_project_summary.csv'
STATS_FILE = ROOT / f'{NETID}_project_stats.csv'
METADATA_FILE = ROOT / f'{NETID}_part1_metadata.json'
print('NetID:', NETID)

NetID: hsaleh5


## Assigned projects

In [2]:
assignments = pd.read_csv(ROOT / 'net2prj.csv', dtype=str)
mine = assignments.loc[assignments['netID'].str.casefold() == NETID.casefold()].copy()
assert len(mine) == 10, f'Expected 10 assigned projects, found {len(mine)}'
display(mine.reset_index(drop=True))

,netID,WoC,GH
0,hsaleh5,vibansal_hapcut2,https://github.com/vibansal/HapCUT2
1,hsaleh5,friendly_heplots,https://github.com/friendly/heplots
2,hsaleh5,elki-project_elki,https://github.com/elki-project/elki
3,hsaleh5,neuroml_neuroml2,https://github.com/NeuroML/NeuroML2
4,hsaleh5,ropensci_gsodr,https://github.com/ropensci/GSODR
5,hsaleh5,modelica_modelica_linearsystems2,https://github.com/modelica/Modelica_LinearSys...
6,hsaleh5,wangbolun300_fast-envelope,https://github.com/wangbolun300/fast-envelope
7,hsaleh5,antvis_g6,https://github.com/antvis/G6
8,hsaleh5,lsstdesc_ccl,https://github.com/LSSTDESC/CCL
9,hsaleh5,dsavransky_exosims,https://github.com/dsavransky/EXOSIMS


## Retrieval

In [3]:
# Set True to rebuild the CSVs from WoC/GitHub. The checked-in output files were
# generated by this exact entry point; False avoids unnecessary repeat API traffic.
RUN_RETRIEVAL = False
if RUN_RETRIEVAL:
    from retrieve_part1 import main
    main()
else:
    print('Using the completed, checked-in Part 1 retrieval artifacts.')

Using the completed, checked-in Part 1 retrieval artifacts.


### Retrieval safeguards

The retrieval client checkpoints records per project, retries transient HTTP failures, uses ten-commit batches, pauses 3.2 seconds between batch requests, verifies SHA-1 uniqueness, and refuses to silently invent missing content. WoC returned 41 `p2c` keys across four projects that had no `commit.tch` record (the local Git objects identify these as annotated-tag/non-commit objects); these were reported and excluded.

## Required Part 1 results

| Project (WoC ID) | WoC commits | WoC authors | Min time (Unix; UTC) | Max time (Unix; UTC) | GitHub stars | GitHub forks | Last GitHub commit |
|---|---:|---:|---|---|---:|---:|---|
| `vibansal_hapcut2` | 588 | 22 | 1405548177<br>2014-07-16 22:02:57Z | 1741933800<br>2025-03-14 06:30:00Z | 233 | 39 | 2025-02-09 |
| `friendly_heplots` | 721 | 8 | 1173790214<br>2007-03-13 12:50:14Z | 1757692802<br>2025-09-12 16:00:02Z | 11 | 5 | 2026-09-13 |
| `elki-project_elki` | 15,704 | 94 | 1121867691<br>2005-07-20 13:54:51Z | 1775749738<br>2026-04-09 15:48:58Z | 831 | 321 | 2026-08-10 |
| `neuroml_neuroml2` | 986 | 18 | 1352295427<br>2012-11-07 13:37:07Z | 1739353578<br>2025-02-12 09:46:18Z | 66 | 26 | 2026-09-02 |
| `ropensci_gsodr` | 2,912 | 18 | 1427153177<br>2015-03-23 23:26:17Z | 1762248461<br>2025-11-04 09:27:41Z | 98 | 19 | 2026-05-17 |
| `modelica_modelica_linearsystems2` | 1,501 | 22 | 1251871200<br>2009-09-02 06:00:00Z | 1761568752<br>2025-10-27 12:39:12Z | 26 | 21 | 2026-09-24 |
| `wangbolun300_fast-envelope` | 488 | 13 | 1559158533<br>2019-05-29 19:35:33Z | 1751287292<br>2025-06-30 12:41:32Z | 64 | 14 | 2025-06-30 |
| `antvis_g6` | 16,355 | 474 | 1486980044<br>2017-02-13 10:00:44Z | 1762258188<br>2025-11-04 12:09:48Z | 12,318 | 1,646 | 2026-09-23 |
| `lsstdesc_ccl` | 8,694 | 190 | 1461938918<br>2016-04-29 14:08:38Z | 1762533598<br>2025-11-07 16:39:58Z | 160 | 87 | 2026-07-28 |
| `dsavransky_exosims` | 4,319 | 66 | 1433177497<br>2015-06-01 16:51:37Z | 1762203232<br>2025-11-03 20:53:52Z | 36 | 49 | 2026-09-29 |

`Min time` and `Max time` are author timestamps from WoC commit records. GitHub stars/forks and the last default-branch commit date are live-repository metadata.

## Validation

In [4]:
summary = pd.read_csv(SUMMARY_FILE, sep=';', dtype={'commit_sha1': str})
stats = pd.read_csv(STATS_FILE, sep=';')
expected_columns = ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']
assert summary.columns.tolist() == expected_columns
assert summary['project_wocid'].nunique() == 10
assert not summary.isna().any().any()
assert not summary.duplicated(['project_wocid', 'commit_sha1']).any()

computed = (summary.groupby('project_wocid', as_index=False)
            .agg(ncommits=('commit_sha1', 'size'),
                 nauthors=('author', 'nunique'),
                 min_time=('time', 'min'),
                 max_time=('time', 'max')))
check = computed.merge(stats, left_on='project_wocid', right_on='Project')
assert (check['ncommits_x'] == check['ncommits_y']).all()
assert (check['nauthors_x'] == check['nauthors_y']).all()
assert (check['min_time'] == check['from']).all()
assert (check['max_time'] == check['to']).all()
print(f'Validated {len(summary):,} commit rows across {summary.project_wocid.nunique()} projects.')
display(stats)

Validated 52,268 commit rows across 10 projects.


,Project,ncommits,nauthors,from,to,nstars,nforks,lastGHCommitDate
0,vibansal_hapcut2,588,22,1405548177,1741933800,233,39,2025-02-09
1,friendly_heplots,721,8,1173790214,1757692802,11,5,2026-09-13
2,elki-project_elki,15704,94,1121867691,1775749738,831,321,2026-08-10
3,neuroml_neuroml2,986,18,1352295427,1739353578,66,26,2026-09-02
4,ropensci_gsodr,2912,18,1427153177,1762248461,98,19,2026-05-17
5,modelica_modelica_linearsystems2,1501,22,1251871200,1761568752,26,21,2026-09-24
6,wangbolun300_fast-envelope,488,13,1559158533,1751287292,64,14,2025-06-30
7,antvis_g6,16355,474,1486980044,1762258188,12318,1646,2026-09-23
8,lsstdesc_ccl,8694,190,1461938918,1762533598,160,87,2026-07-28
9,dsavransky_exosims,4319,66,1433177497,1762203232,36,49,2026-09-29


### GitHub comparison

GitHub's REST commit total describes the current default branch, while WoC's `p2c` map can include commits from other branches and historical refs. Therefore the two totals are useful as a reasonableness check but are not expected to match exactly. The observed differences are consistent with those different scopes and with GitHub activity after the WoC snapshot/indexing point.

In [5]:
meta = json.loads(METADATA_FILE.read_text(encoding='utf-8'))
comparison = pd.DataFrame(meta['projects'])[[
    'Project', 'ncommits', 'nauthors', 'github_default_branch_commits',
    'github_contributors', 'lastGHCommitDate'
]]
display(comparison)

,Project,ncommits,nauthors,github_default_branch_commits,github_contributors,lastGHCommitDate
0,vibansal_hapcut2,588,22,555,12,2025-02-09
1,friendly_heplots,721,8,853,6,2026-09-13
2,elki-project_elki,15704,94,10549,58,2026-08-10
3,neuroml_neuroml2,986,18,932,12,2026-09-02
4,ropensci_gsodr,2912,18,2740,11,2026-05-17
5,modelica_modelica_linearsystems2,1501,22,1154,12,2026-09-24
6,wangbolun300_fast-envelope,488,13,481,10,2025-06-30
7,antvis_g6,16355,474,5620,223,2026-09-23
8,lsstdesc_ccl,8694,190,3992,87,2026-07-28
9,dsavransky_exosims,4319,66,3921,39,2026-09-29


## Output preview

In [6]:
print('Delimiter: semicolon (;)')
print('Columns:', summary.columns.tolist())
display(summary.head(5))

Delimiter: semicolon (;)
Columns: ['project_wocid', 'commit_sha1', 'author', 'time', 'commit message']


,project_wocid,commit_sha1,author,time,commit message
0,vibansal_hapcut2,0096d0226fd452ccc406c14b00e6f23abfa111c4,Peter Edge <edge.peterj@gmail.com>,1475273660,Update hapcut2.c\n\nremoved --v as an option f...
1,vibansal_hapcut2,00a663d51502f56bad02deb82d5f4e48a4ef8736,Peter Edge <edge.peterj@gmail.com>,1459566607,added scripts for new versions of known and un...
2,vibansal_hapcut2,0121162b713abd259815c98943b04ab1d423d73f,Peter Edge <edge.peterj@gmail.com>,1493502480,Lifted limit on number of nodes in iterative l...
3,vibansal_hapcut2,01af9003507d608df96b65bd65f7b6e9e8940284,Peter Edge <edge.peterj@gmail.com>,1462382028,fixed conflict\n
4,vibansal_hapcut2,024e1b905a54cf0e68e4d05c45dea00aa905be71,Vikas Bansal <vikas115@gmail.com>,1555725262,working implementation of phased/unphased geno...
